<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/Copy_of_tapvidmv_scenepic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from etils import ecolab
from google3.pyglib import gfile
import os
import numpy as np
import tensorflow as tf
import mediapy as media

import trimesh
from colabtools import fileedit

def inv(mat):
  rot = np.swapaxes(mat[..., :3, :3], -1, -2).astype(np.float32)
  trans = -rot @ mat[..., :3, 3:]
  return np.concatenate((rot, trans), axis=-1)

def cat(mat):
  if mat.shape[-2:] == (3, 4):
    row = np.ones_like(mat[..., :1, :]) * np.array([0.0, 0.0, 0.0, 1.0])
    mat = np.concatenate((mat, row), axis=-2)
  return mat

In [ ]:
base_path = '/cns/lu-d/home/gdm-4d-scenes/data/tapvidmv/'

In [ ]:
splits = gfile.ListDir(base_path)
print(splits)

In [ ]:
split='droid'

In [ ]:
seqs = list(filter(lambda x: gfile.IsDirectory(x), gfile.Glob(os.path.join(base_path, split, '*'))))

In [ ]:
seq_idx = 5
seq = seqs[seq_idx]

In [ ]:
seq

In [ ]:
views = list(filter(lambda x: gfile.IsDirectory(x), gfile.Glob(os.path.join(seq, '*'))))

In [ ]:
views

In [ ]:
view_idx = 0

def load_data(view_idx):
  depth_file = os.path.join(views[view_idx],'depth.npy')
  extrinsics_w2c_file = os.path.join(views[view_idx],'extrinsics_w2c.npy')
  rgb_file = os.path.join(views[view_idx],'images_jpeg_bytes.npy')
  intrinsics_file = os.path.join(views[view_idx],'intrinsics.npy')
  # visibility_file = os.path.join(views[view_idx],'visibility.npy')

  depth = np.load(gfile.GFile(depth_file), allow_pickle=True)
  rgb = np.load(gfile.GFile(rgb_file), allow_pickle=True)
  rgb = [np.array(tf.io.decode_jpeg(x.tobytes())) for x in rgb]
  vid = np.stack(rgb, axis=0)
  intrinsics = np.load(gfile.GFile(intrinsics_file), allow_pickle=True)
  extrinsics_w2c = np.load(gfile.GFile(extrinsics_w2c_file), allow_pickle=True)
  fx, fy, cx, cy = intrinsics
  K = np.array([[fx, 0., cx], [0., fy, cy], [0., 0., 1.]])
  h, w = rgb[0].shape[:2]
  y, x = np.meshgrid(np.linspace(0.5, h-0.5, h), np.linspace(0.5, w-0.5, w), indexing='ij')
  xy1 = np.stack([x, y, np.ones_like(x)], axis=-1)

  ok_depth = depth != float('inf')

  xyz = [(xy1 @ np.linalg.inv(K).T)[ok] * d[ok][...,None] for ok, d in zip(ok_depth, depth)]
  rgb = [x[ok] for ok, x in zip(ok_depth, rgb)]

  extrinsics_c2w = inv(extrinsics_w2c)
  xyz_world = [x @ c2w[:3,:3].T + c2w[:3, 3:].T for x, c2w in zip(xyz, extrinsics_c2w)]
  return vid, xyz_world, rgb, extrinsics_w2c

In [ ]:
vid_0, xyz_world_0, rgb_0, extrinsics_w2c_0 = load_data(0)

In [ ]:
len(xyz_world_0)

In [ ]:
media.show_video(vid_0)

In [ ]:
len(rgb_0)

In [ ]:
len(vid_0)

In [ ]:
vid_1, xyz_world_1, rgb_1, extrinsics_w2c_1 = load_data(1)

In [ ]:
vid_2, xyz_world_2, rgb_2, extrinsics_w2c_2 = load_data(2)

In [ ]:
with ecolab.adhoc('tapvidmv', reload='google3.gdm.vision.superdense.product.colab.scenepic'):
  from google3.gdm.vision.superdense.product.colab import scenepic as sp_utils

In [ ]:
tracks_xyz = np.load(gfile.GFile(os.path.join(seq,'tracks_xyz.npy')), allow_pickle=True)

In [ ]:
vel_xyz = tracks_xyz[1:]-tracks_xyz[:-1]

In [ ]:
accel_xyz = vel_xyz[1:]-vel_xyz[:-1]

In [ ]:
mean_accel = np.linalg.norm(accel_xyz, axis=-1).mean(axis=0)

In [ ]:
thresh = np.median(mean_accel) * 10

In [ ]:
ok_points = mean_accel < thresh

In [ ]:
stride=1

In [ ]:
t_start=30
t_end=40

In [ ]:
media.show_image(vid_0[30])

In [ ]:
xyz_world_0_subsampled = [xyz.reshape(vid_0.shape[1],vid_0.shape[2],3)[::stride,::stride].reshape(-1,3) for xyz in xyz_world_0]
rgb_0_subsampled = [rgb.reshape(vid_0.shape[1],vid_0.shape[2],3)[::stride,::stride].reshape(-1,3).astype(np.float32)/255 for rgb in rgb_0]

xyz_world_1_subsampled = [xyz.reshape(vid_0.shape[1],vid_0.shape[2],3)[::stride,::stride].reshape(-1,3) for xyz in xyz_world_1]
rgb_1_subsampled = [rgb.reshape(vid_0.shape[1],vid_0.shape[2],3)[::stride,::stride].reshape(-1,3).astype(np.float32)/255 for rgb in rgb_1]

xyz_world_2_subsampled = [xyz.reshape(vid_0.shape[1],vid_0.shape[2],3)[::stride,::stride].reshape(-1,3) for xyz in xyz_world_2]
rgb_2_subsampled = [rgb.reshape(vid_0.shape[1],vid_0.shape[2],3)[::stride,::stride].reshape(-1,3).astype(np.float32)/255 for rgb in rgb_2]

In [ ]:
xyz_world_0_subsampled = xyz_world_0_subsampled[t_start:t_end]
rgb_0_subsampled = rgb_0_subsampled[t_start:t_end]
xyz_world_1_subsampled = xyz_world_1_subsampled[t_start:t_end]
rgb_1_subsampled = rgb_1_subsampled[t_start:t_end]
xyz_world_2_subsampled = xyz_world_2_subsampled[t_start:t_end]
rgb_2_subsampled = rgb_2_subsampled[t_start:t_end]

In [ ]:
len(xyz_world_0_subsampled)

In [ ]:
xyz_world_0_subsampled = [x[:1] if idx!=len(xyz_world_0_subsampled)-1 else x for idx, x in enumerate(xyz_world_0_subsampled)]
rgb_0_subsampled = [x[:1] if idx!=len(xyz_world_0_subsampled)-1 else x for idx, x in enumerate(rgb_0_subsampled)]
xyz_world_1_subsampled = [x[:1] if idx!=len(xyz_world_0_subsampled)-1 else x for idx, x in enumerate(xyz_world_1_subsampled)]
rgb_1_subsampled = [x[:1] if idx!=len(xyz_world_0_subsampled)-1 else x for idx, x in enumerate(rgb_1_subsampled)]
xyz_world_2_subsampled = [x[:1] if idx!=len(xyz_world_0_subsampled)-1 else x for idx, x in enumerate(xyz_world_2_subsampled)]
rgb_2_subsampled = [x[:1] if idx!=len(xyz_world_0_subsampled)-1 else x for idx, x in enumerate(rgb_2_subsampled)]


In [ ]:
# xyz_world_0_subsampled = [xyz[::stride] for xyz in xyz_world_0]
# rgb_0_subsampled = [rgb[::stride].astype(np.float32)/255 for rgb in rgb_0]
# xyz_world_1_subsampled = [xyz[::stride] for xyz in xyz_world_1]
# rgb_1_subsampled = [rgb[::stride].astype(np.float32)/255 for rgb in rgb_1]

In [ ]:
pcl = sp_utils.plot_point_cloud(
    [np.concatenate([xyz_0, xyz_1, xyz_2], axis=0) for xyz_0, xyz_1, xyz_2 in zip(xyz_world_0_subsampled, xyz_world_1_subsampled, xyz_world_2_subsampled)],
    [np.concatenate([rgb_0, rgb_1, rgb_2], axis=0) for rgb_0, rgb_1, rgb_2 in zip(rgb_0_subsampled, rgb_1_subsampled, rgb_2_subsampled)],
    # [np.concatenate([xyz_0, xyz_1], axis=0) for xyz_0, xyz_1 in zip(xyz_world_0_subsampled, xyz_world_1_subsampled)],
    # [np.concatenate([rgb_0, rgb_1], axis=0) for rgb_0, rgb_1 in zip(rgb_0_subsampled, rgb_1_subsampled)],
    tracks_xyz[t_start:t_end,ok_points],
    pred_cam_poses=np.stack([extrinsics_w2c_0, extrinsics_w2c_1, extrinsics_w2c_2], axis=1)[t_start:t_end],
    # gt_cam_poses=np.stack([extrinsics_w2c_0, extrinsics_w2c_1], axis=1)[t_start:t_end],
    frustum_depth=0.1,
    frustum_thickness=0.005,
    bg_sphere_radius=0.002,
    track_sphere_radius=0.005,
    track_thickness=0.001,
    # bg_sphere_radius=0.1,
)

In [ ]:
with open('pcl.html', 'w') as f:
  f.write(pcl)
!fileutil cp -f pcl.html /x20/users/ir/irocco/ > /dev/null 2>&1
print('https://x20web.corp.google.com/users/ir/irocco/pcl.html')

In [ ]:
_=trimesh.PointCloud(tracks_xyz.reshape(-1,3)).export('droid_tracks.ply')

In [ ]:
fileedit.download_file('droid_tracks.ply', ephemeral=True)

In [ ]:
import glob
import shutil
# @title Generate gif/png of 4D visualization
# @markdown First generate a zip file using the 🔴 record button (in previous cell). <br><br>
# @markdown [local] The faster alternative is to generate the gif locally (tested on MAC): <br>
# @markdown - follow [this link](https://yaqs.corp.google.com/eng/q/5129125029478400#a5707702298738688) to install brew / ffmpeg on mac
# @markdown - extract zip and cd to png folder
# @markdown - running the following command will crop the images to a tight bounding box and render the gif at 10 fps
# @markdown ```
# @markdown CROP_DATA=$(ffmpeg -loglevel info -i frame_%05d.png -vf "alphaextract,bbox=min_val=128" -f null - 2>&1 | awk 'BEGIN{mx1=99999;my1=99999;mx2=0;my2=0} /Parsed_bbox_/ { for(i=1;i<=NF;i++){ if($i ~ /^x1:/){x1=$i; gsub("x1:","",x1); if(x1<mx1)mx1=x1} else if($i ~ /^x2:/){x2=$i; gsub("x2:","",x2); if(x2>mx2)mx2=x2} else if($i ~ /^y1:/){y1=$i; gsub("y1:","",y1); if(y1<my1)my1=y1} else if($i ~ /^y2:/){y2=$i; gsub("y2:","",y2); if(y2>my2)my2=y2} } } END{if(mx1==99999 || my1==99999){print "ERROR: No bbox data found"} else {W=mx2-mx1;H=my2-my1;printf "crop=%s:%s:%s:%s", W, H, mx1, my1}}') && echo "Detected crop: $CROP_DATA" && [[ "$CROP_DATA" != ERROR* ]] && ffmpeg -framerate 10 -i frame_%05d.png -filter_complex "[0:v] $CROP_DATA, split [a][b]; [a] palettegen=reserve_transparent=1 [p]; [b][p] paletteuse=alpha_threshold=128" output.gif
# @markdown ```
# @markdown [colab] Run this cell and wait until 'Upload zip file from interactive visualizer' window shows up to upload the zip file
FORMAT = 'gif'  # @param ['gif', 'png']

print('Upload zip file from interactive visualizer')
for c in glob.glob('Canvas-*'):
  shutil.rmtree(c)
zip_bytes = fileedit.upload_files()
with open('ims.zip', 'wb') as f:
  f.write(list(zip_bytes.values())[0])
is_rgba = True



In [ ]:
for im in sp_utils.zip_to_gifs('ims.zip', os.path.basename(seq), FORMAT, is_rgba):
  fileedit.download_file(im, ephemeral=True)
  media.show_video(media.read_video(im), codec='gif', fps=10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Your color dictionary with 0-255 RGB values
colors = {
    "Black": (0, 0, 0), "White": (255, 255, 255), "Red": (255, 0, 0),
    "Maroon": (128, 0, 0), "Pink": (255, 200, 220), "Brown": (170, 110, 40),
    "Orange": (255, 150, 0), "Coral": (255, 215, 180), "Olive": (128, 128, 0),
    "Yellow": (255, 235, 0), "Beige": (255, 250, 200), "Lime": (190, 255, 0),
    "Green": (0, 190, 0), "Mint": (170, 255, 195), "Teal": (0, 128, 128),
    "Cyan": (100, 255, 255), "Navy": (0, 0, 128), "Blue": (67, 133, 255),
    "Purple": (130, 0, 150), "Lavender": (230, 190, 255), "Magenta": (255, 0, 255),
    "Gray": (128, 128, 128)
}

# Create a grid layout for the colors
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 4)
ax.set_ylim(0, 6)

for i, (name, rgb) in enumerate(colors.items()):
    # Calculate grid coordinates (4 columns)
    col = i % 4
    row = 5 - (i // 4)

    # Normalize 0-255 integers to 0.0-1.0 floats for Matplotlib
    normalized_rgb = np.array(rgb) / 255.0

    # Draw a colored rectangle
    rect = plt.Rectangle((col + 0.1, row + 0.1), 0.8, 0.8, color=normalized_rgb, ec="black", lw=1)
    ax.add_patch(rect)

    # Add text label (adjusting text color based on background brightness for visibility)
    text_color = "white" if sum(rgb) < 350 else "black"
    ax.text(col + 0.5, row + 0.5, name, va="center", ha="center", weight="bold", color=text_color)

plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
with ecolab.adhoc():
  import google3.third_party.projectaria_tools.core.mps as mps
  from google3.third_party.projectaria_tools.core.sensor_data import TimeDomain
  from google3.third_party.projectaria_tools.core.mps.utils import filter_points_from_confidence
  from google3.third_party.projectaria_tools.core import data_provider
  from google3.third_party.projectaria_tools.core.mps.utils import get_nearest_pose
  from google3.third_party.projectaria_tools.core.stream_id import StreamId
  from google3.third_party.projectaria_tools.core.calibration import (
      device_calibration_from_json_string,
      distort_by_calibration,
      get_linear_camera_calibration,
  )